# Colab Session C — KB v1 vs v2 A/B Evaluation

**Runtime:** Any GPU (T4 fine)
**Est. time:** ~45 min
**Prerequisites:** Session A (KB v2 on Drive) AND Session B (results) must be complete.

**Task:** Run identical 97-question evaluation against both KB v1 and KB v2.  
Switch to KB v2 as default only if it wins by ≥0.03 keyword coverage.


In [ ]:
GITHUB_REPO   = "https://github.com/kbssrikar7/final_project.git"
GITHUB_BRANCH = "main"
DRIVE_BASE    = "/content/drive/MyDrive/healthcare_qa"
PROJECT_DIR   = "/content/project"
print("Configuration loaded")

In [ ]:
# C1: Setup + sync KB v2 from Drive
import subprocess, shutil, os
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", GITHUB_BRANCH], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Sync both KBs
for src, dst in [
    (f"{DRIVE_BASE}/knowledge_base",    f"{PROJECT_DIR}/data/knowledge_base"),
    (f"{DRIVE_BASE}/knowledge_base_v2", f"{PROJECT_DIR}/data/knowledge_base_v2"),
    (f"{DRIVE_BASE}/models",             f"{PROJECT_DIR}/models"),
]:
    if os.path.exists(src) and not os.path.exists(dst):
        print(f"Syncing {src} → {dst} ...")
        shutil.copytree(src, dst)
        print("Done")
    elif not os.path.exists(src):
        print(f"WARNING: {src} not found on Drive")

print("Setup complete")

In [ ]:
# C2: Eval with KB v1
import subprocess, time, json
from pathlib import Path

print("Evaluating with KB v1 (standard) ...")
t0 = time.time()
subprocess.run([
    "python3", "evaluation/run_paper_eval.py",
    "--mode", "metrics", "--n", "97", "--model", "tinyllama",
], check=True)
v1_elapsed = time.time() - t0

# Copy result to v1-specific filename
shutil.copy2("evaluation/results/metrics_full_tinyllama.json",
             "evaluation/results/metrics_kbv1_tinyllama.json")

v1 = json.loads(Path("evaluation/results/metrics_kbv1_tinyllama.json").read_text())
kw_v1 = v1.get('keyword_coverage_mean', v1.get('keyword_coverage', 0))
print(f"KB v1 keyword coverage: {kw_v1:.4f}  ({v1_elapsed/60:.1f} min)")

In [ ]:
# C3: Eval with KB v2 — switch persist_directory env var
import subprocess, time, json, os, shutil
from pathlib import Path

env_v2 = os.environ.copy()
env_v2["KB_PERSIST_DIR"] = f"{PROJECT_DIR}/data/knowledge_base_v2"
env_v2["CHROMA_COLLECTION"] = "medical_knowledge_v2"

print("Evaluating with KB v2 (RecursiveSentenceChunker) ...")
t0 = time.time()
subprocess.run([
    "python3", "evaluation/run_paper_eval.py",
    "--mode", "metrics", "--n", "97", "--model", "tinyllama",
], env=env_v2, check=True)
v2_elapsed = time.time() - t0

shutil.copy2("evaluation/results/metrics_full_tinyllama.json",
             "evaluation/results/metrics_kbv2_tinyllama.json")

v2 = json.loads(Path("evaluation/results/metrics_kbv2_tinyllama.json").read_text())
kw_v2 = v2.get('keyword_coverage_mean', v2.get('keyword_coverage', 0))
print(f"KB v2 keyword coverage: {kw_v2:.4f}  ({v2_elapsed/60:.1f} min)")

In [ ]:
# C4: Decision record
import json
from pathlib import Path

delta = kw_v2 - kw_v1
decision = "v2" if delta >= 0.03 else "v1"

record = {
    "kw_v1": round(kw_v1, 4),
    "kw_v2": round(kw_v2, 4),
    "delta": round(delta, 4),
    "threshold": 0.03,
    "decision": decision,
    "reason": "v2 wins by >=0.03 keyword coverage" if decision == "v2"
              else "v2 does not improve by threshold — keep v1",
}

out_path = Path("evaluation/results/kb_v1_vs_v2.json")
out_path.write_text(json.dumps(record, indent=2))

print("=" * 50)
print(f"KB v1 keyword coverage: {kw_v1:.4f}")
print(f"KB v2 keyword coverage: {kw_v2:.4f}")
print(f"Delta: {delta:+.4f}")
print(f"Decision: USE {decision.upper()}'s KB as production default")
print("=" * 50)
print(f"Written to: {out_path}")
print()
print("ACTION REQUIRED (local):")
if decision == "v2":
    print("  Edit config/settings.py: persist_directory = 'data/knowledge_base_v2'")
    print("  Edit config/settings.py: collection_name = 'medical_knowledge_v2'")
else:
    print("  Keep config/settings.py unchanged (KB v1 stays)")

In [ ]:
# C5: Push decision record to Drive + GitHub
import subprocess, shutil, os

# Drive
shutil.copy2("evaluation/results/kb_v1_vs_v2.json",
             f"{DRIVE_BASE}/evaluation_results/kb_v1_vs_v2.json")
print("Decision record pushed to Drive")

# GitHub
try:
    subprocess.run(["git", "config", "user.email", "colab@healthcare-qa"], check=True)
    subprocess.run(["git", "config", "user.name", "Colab Session C"], check=True)
    subprocess.run(["git", "add",
                    "evaluation/results/kb_v1_vs_v2.json",
                    "evaluation/results/metrics_kbv1_tinyllama.json",
                    "evaluation/results/metrics_kbv2_tinyllama.json"], check=True)
    subprocess.run(["git", "commit", "-m",
                    f"eval: KB v1 vs v2 A/B — decision={decision} (delta={delta:+.4f})"],
                   check=True)
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], check=True)
    print("Results committed and pushed to GitHub")
except Exception as e:
    print(f"WARNING: Git push failed: {e}. Results are safe on Drive.")

print("\nAll 3 sessions complete. Pull results locally, then proceed to Phase 5 (paper).")